# S3.4 — Writing Data — Modes, Formats, Options
**Date completed:** September 2026  
**Status:** Completed  
**Key concept:** Write mode determines pipeline idempotency

In [0]:
# ============================================================
# Cell 2 — Write Modes: Understanding Duplicate Data
#
# Goal:
# 1. Understand append, overwrite, ignore and errorifexists.
# 2. Simulate duplicate creation during repeated append loads.
# 3. Understand full-reload overwrite behaviour.
# 4. Compare results using row counts and duplicate checks.
#
# IMPORTANT:
# This cell SIMULATES write behaviour using DataFrames.
# No actual storage write is performed.
#
# Actual file/table writing will be covered separately.
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

print("=== WRITE MODES DEMONSTRATION ===")


# ------------------------------------------------------------
# Step 1 — Create sample Silver data
# ------------------------------------------------------------

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("order_value", DoubleType(), True)
])

data = [
    (1001, 201, "Delhi", 2500.0),
    (1002, 202, "Mumbai", 1800.0),
    (1003, 203, "Pune", 3200.0),
    (1004, 204, "Chennai", 4100.0),
    (1005, 205, "Delhi", 950.0)
]

df_silver = spark.createDataFrame(
    data,
    schema=schema
)

print("\n=== SOURCE DATA ===")

df_silver.show()

print(f"Source records: {df_silver.count()}")


# ------------------------------------------------------------
# Step 2 — Understand the four write modes
# ------------------------------------------------------------

print("\n=== WRITE MODE COMPARISON ===")

print("""
APPEND:
Adds incoming records to existing data.
Repeated ingestion of the same records may create duplicates.

OVERWRITE:
Replaces existing data within the write operation's target scope.
Useful for controlled full-refresh processing.

IGNORE:
Skips the write if the target already exists.

ERRORIFEXISTS:
Raises an error if the target already exists.
This is the default write mode.
""")


# ------------------------------------------------------------
# Step 3 — Simulate two pipeline executions
#
# Assume the same three records are received twice.
# ------------------------------------------------------------

first_run = spark.createDataFrame(
    data[:3],
    schema=schema
)

second_run = spark.createDataFrame(
    data[:3],
    schema=schema
)


# ------------------------------------------------------------
# Step 4 — Simulate APPEND
#
# unionByName() combines the two datasets.
# This represents a naive append without deduplication.
# ------------------------------------------------------------

print("\n=== APPEND SIMULATION ===")

df_append_result = first_run.unionByName(second_run)

df_append_result.show()

print(
    f"Records after repeated append: "
    f"{df_append_result.count()}"
)


# ------------------------------------------------------------
# Step 5 — Identify duplicate order IDs
# ------------------------------------------------------------

print("\n=== DUPLICATE VALIDATION ===")

df_append_result.groupBy("order_id") \
    .agg(F.count("*").alias("occurrences")) \
    .filter(F.col("occurrences") > 1) \
    .show()


# ------------------------------------------------------------
# Step 6 — Simulate OVERWRITE
#
# Second full batch replaces the previous full batch.
# This is an in-memory conceptual demonstration.
# ------------------------------------------------------------

print("\n=== OVERWRITE SIMULATION ===")

df_overwrite_result = second_run

df_overwrite_result.show()

print(
    f"Records after simulated overwrite: "
    f"{df_overwrite_result.count()}"
)

# ------------------------------------------------------------
# Step 7 — Final comparison
# ------------------------------------------------------------

print("\n=== FINAL COMPARISON ===")

print("Original batch         : 3 records")
print(f"Repeated naive append  : {df_append_result.count()} records")
print(f"Full overwrite result  : {df_overwrite_result.count()} records")

print()
print("Learning: APPEND does not automatically prevent duplicate records.")
print("Learning: OVERWRITE replaces data within the specified write scope.")
print("Learning: Real Delta writes will be tested in a separate practical.")

print("\n=== CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# Cell 3 — Writing Data: Formats, Options and File Management
#
# Goal:
# 1. Understand CSV, JSON, Parquet and Delta write syntax.
# 2. Understand commonly used writing options.
# 3. Understand compression choices.
# 4. Demonstrate coalesce() before writing.
# 5. Understand storage partitioning using partitionBy().
# 6. Use the actual Databricks catalog and schema.
#
# Confirmed environment:
# Catalog : workspace
# Schema  : default
#
# IMPORTANT:
# This is a conceptual demonstration.
# No physical files or tables are written in this cell.
#
# Actual file writing will require an authorised storage path.
# ============================================================

import pyspark.sql.functions as F

print("=== WRITING FORMAT OPTIONS ===")
print()


# ------------------------------------------------------------
# Step 1 — Create sample RetailPulse data
# ------------------------------------------------------------

df_write_demo = spark.createDataFrame(
    [
        (1001, "Delhi",   2500.0, "2026-09-28"),
        (1002, "Mumbai",  1800.0, "2026-09-28"),
        (1003, "Pune",    3200.0, "2026-09-27"),
        (1004, "Chennai", 4100.0, "2026-09-27"),
        (1005, "Delhi",    950.0, "2026-09-26")
    ],
    [
        "order_id",
        "city",
        "order_value",
        "order_date"
    ]
)

# Convert order_date from StringType to DateType
df_write_demo = df_write_demo.withColumn(
    "order_date",
    F.to_date("order_date")
)

print("=== SAMPLE DATA ===")

df_write_demo.show(truncate=False)

print("=== SOURCE SCHEMA ===")

df_write_demo.printSchema()

print(f"Source row count: {df_write_demo.count()}")


# ------------------------------------------------------------
# Step 2 — CSV Writing Syntax
#
# header=true : Include column names in the output.
# sep=","     : Use comma as the field delimiter.
#
# output_path must point to authorised storage.
# This code only prints the syntax.
# ------------------------------------------------------------

print("\n=== CSV WRITE SYNTAX ===")

print("""
(
    df_write_demo.write
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ",")
    .csv(output_path)
)
""")


# ------------------------------------------------------------
# Step 3 — JSON Writing Syntax
#
# JSON supports structured and nested data.
# This code only prints the syntax.
# ------------------------------------------------------------

print("\n=== JSON WRITE SYNTAX ===")

print("""
(
    df_write_demo.write
    .mode("overwrite")
    .json(output_path)
)
""")


# ------------------------------------------------------------
# Step 4 — Parquet Writing Syntax
#
# Parquet is a columnar file format.
# Snappy is the standard default compression codec.
#
# This code only prints the syntax.
# ------------------------------------------------------------

print("\n=== PARQUET WRITE SYNTAX ===")

print("""
(
    df_write_demo.write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(output_path)
)
""")


# ------------------------------------------------------------
# Step 5 — Delta Writing Syntax
#
# Actual Databricks environment:
#
# Catalog : workspace
# Schema  : default
#
# Proposed learning table:
# workspace.default.s34_orders_demo
#
# saveAsTable() creates or writes a registered table.
#
# errorifexists prevents accidental replacement of an
# existing table during an initial creation attempt.
#
# This code only prints the syntax.
# No table will be created.
# ------------------------------------------------------------

print("\n=== DELTA WRITE SYNTAX ===")

print("""
(
    df_write_demo.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable("workspace.default.s34_orders_demo")
)
""")

print("Catalog : workspace")
print("Schema  : default")
print("Table   : s34_orders_demo")

print("""
Note:
This table has not been created.

Actual execution requires the appropriate permissions
and an available target table name.
""")


# ------------------------------------------------------------
# Step 6 — Storage Partitioning
#
# partitionBy() organises output files into directories
# based on the values of a selected column.
#
# Use only when physical partitioning is justified.
#
# This code only prints the syntax.
# ------------------------------------------------------------

print("\n=== STORAGE PARTITIONING SYNTAX ===")

print("""
(
    df_write_demo.write
    .mode("overwrite")
    .partitionBy("order_date")
    .option("compression", "snappy")
    .parquet(output_path)
)
""")

print("Illustrative output directory structure:")

print("""
orders/
    order_date=2026-09-26/
    order_date=2026-09-27/
    order_date=2026-09-28/
""")

print("""
Important:
partitionBy() controls the physical storage layout.

It is different from DataFrame repartition() and coalesce().
""")


# ------------------------------------------------------------
# Step 7 — Understand coalesce() Before Writing
#
# coalesce(1) reduces processing partitions.
#
# It does NOT write files.
# It should not be applied blindly to large datasets.
# ------------------------------------------------------------

print("\n=== COALESCE BEFORE WRITE ===")

# Count observed non-empty input partitions.
# Note: This is not necessarily the configured partition count.

original_partitions = (
    df_write_demo
    .select(F.spark_partition_id().alias("partition_id"))
    .distinct()
    .count()
)

print(
    f"Observed non-empty input partitions: "
    f"{original_partitions}"
)

# Reduce processing partitions to one
df_coalesced = df_write_demo.coalesce(1)

coalesced_partitions = (
    df_coalesced
    .select(F.spark_partition_id().alias("partition_id"))
    .distinct()
    .count()
)

print(
    f"Observed partitions after coalesce(1): "
    f"{coalesced_partitions}"
)

# Verify that coalesce did not remove any rows
print(f"Original row count : {df_write_demo.count()}")
print(f"Coalesced row count: {df_coalesced.count()}")

print("""
Important:
- coalesce(1) reduces execution parallelism.
- It does not remove data rows.
- It does not itself create a physical file.
- It does not universally guarantee one output file.
- Avoid applying it blindly to large datasets.
- Evaluate Databricks Optimized Writes for Delta workloads.
""")


# ------------------------------------------------------------
# Step 8 — Compression Comparison
#
# Different compression codecs balance:
# - Storage size
# - CPU consumption
# - Compression speed
# - Decompression speed
#
# There is no universally fastest codec for all datasets.
# ------------------------------------------------------------

print("\n=== COMPRESSION OPTIONS ===")

print(
    f"{'Codec':<12}"
    f"{'Characteristics':<36}"
    f"{'Typical Consideration'}"
)

print("-" * 90)

compression_info = [
    (
        "snappy",
        "Fast compression/decompression",
        "General-purpose Parquet"
    ),
    (
        "gzip",
        "Often higher compression, more CPU",
        "When storage size matters"
    ),
    (
        "lz4",
        "Low-latency compression",
        "Speed-oriented workloads"
    ),
    (
        "zstd",
        "Configurable compression trade-off",
        "Evaluate size versus CPU"
    )
]

for codec, characteristics, usage in compression_info:

    print(
        f"{codec:<12}"
        f"{characteristics:<36}"
        f"{usage}"
    )


# ------------------------------------------------------------
# Step 9 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("CSV        : Text format supporting headers and delimiters.")
print("JSON       : Text format supporting structured/nested data.")
print("Parquet    : Columnar storage with compression.")
print("Delta      : Transactional table format using Parquet data files.")
print("coalesce() : Reduces DataFrame processing partitions.")
print("partitionBy: Organises written data using column values.")

print()

print("=== CONFIRMED DATABRICKS ENVIRONMENT ===")

print("Catalog        : workspace")
print("Schema         : default")
print("Learning table : workspace.default.s34_orders_demo")

print()

print("IMPORTANT:")
print("CSV, JSON and Parquet require an authorised output path.")
print("The Delta table example uses our confirmed namespace.")
print("No persistent data has been written by this cell.")

print("\n=== CELL 3 COMPLETED ===")

In [0]:
# ============================================================
# S3.4 — Storage Discovery
# No catalog, schema or table will be created.
# ============================================================

print("=== CURRENT CATALOG AND SCHEMA ===")

spark.sql("""
    SELECT
        current_catalog() AS current_catalog,
        current_schema() AS current_schema
""").show(truncate=False)

print("=== AVAILABLE CATALOGS ===")

spark.sql("SHOW CATALOGS").show(truncate=False)

# S3.4 — Writing Data: Final Key Takeaways

## 1. What Is DataFrameWriter?

`df.write` is used to persist a Spark DataFrame into a storage format or a registered table.

Basic syntax:

```python
df.write \
    .format("delta") \
    .mode("append") \
    .save(output_path)
```

- `format()` defines the storage format.
- `mode()` defines how existing target data is handled.
- `option()` configures format-specific writing behaviour.
- `save()` writes data to a specified storage path.
- `saveAsTable()` writes a registered table.

---

## 2. Write Modes

| Mode | Behaviour | Important Consideration |
|---|---|---|
| overwrite | Replaces existing data within the write scope | Validate the incoming dataset before replacement |
| append | Adds new records | Reprocessing the same batch may produce duplicates |
| ignore | Skips writing if the target already exists | Existing target remains unchanged |
| errorifexists | Raises an error if the target already exists | Default write mode; `error` is also accepted |

### Important Learning

Append does not automatically guarantee unique records.

Overwrite does not automatically guarantee that the incoming data is correct.

Delta Lake provides ACID transactions. A failed Delta transaction does not commit a partial overwrite. This protection should not be assumed for ordinary CSV or Parquet file writes.

---

## 3. Writing Data in Different Formats

### CSV

```python
df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .csv(output_path)
```

### JSON

```python
df.write \
    .mode("overwrite") \
    .json(output_path)
```

### Parquet

```python
df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(output_path)
```

### Delta — Path-Based

```python
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)
```

### Delta — Registered Table

```python
df.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("workspace.default.s34_orders_demo")
```

The final example uses our actual Databricks catalog and schema. It requires appropriate table-creation permissions and an available table name.

---

## 4. Compression Options

| Codec | General Characteristics |
|---|---|
| snappy | Fast and widely used; default Parquet compression |
| gzip | Often smaller output, with higher CPU cost |
| lz4 | Low-latency compression |
| zstd | Configurable size-versus-performance trade-off |

There is no universally fastest compression codec for every workload.

Choose compression based on file size, processing speed, CPU usage and storage requirements.

---

## 5. coalesce() Before Writing

```python
df_coalesced = df.coalesce(1)
```

- Reduces the number of processing partitions.
- May reduce the number of output files in suitable write scenarios.
- Does not itself save any data.
- Does not guarantee exactly one physical output file in every writing scenario.
- Can reduce parallelism and create a processing bottleneck.

Do not blindly apply `coalesce(1)` to large datasets.

For Delta workloads, evaluate Databricks Optimized Writes and other supported file-management features.

---

## 6. partitionBy() — Storage Partitioning

Example:

```python
df.write \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .parquet(output_path)
```

Illustrative directory structure:

orders/
    order_date=2026-09-26/
    order_date=2026-09-27/
    order_date=2026-09-28/

### Important Difference

| Operation | Purpose |
|---|---|
| repartition() | Changes DataFrame execution partitioning, introducing a shuffle |
| coalesce() | Reduces DataFrame execution partitions without a full shuffle in the usual case |
| partitionBy() | Organises persisted output according to column values |

Storage partitioning can enable partition pruning during filtered reads.

Not every large table requires physical partitioning. The appropriate data layout depends on workload, query patterns and table characteristics.

We will explore physical storage and ADLS Gen2 integration in S8.

---

## 7. Our Actual Databricks Environment

We executed storage discovery and confirmed:

| Property | Actual Value |
|---|---|
| Current catalog | workspace |
| Current schema | default |
| Current namespace | workspace.default |
| Compute environment | Databricks Serverless |

### Available Catalogs

- dbacademy
- healthcare_dev
- samples
- system
- workspace

The existence of a catalog does not automatically confirm write permissions.

### Unity Catalog Naming Convention

```text
catalog.schema.table
```

Our proposed learning table:

```text
workspace.default.s34_orders_demo
```

Our planned future RetailPulse production naming example:

```text
retailpulse.silver.orders
```

The RetailPulse production catalog and schema have not yet been created.

---

## 8. Where Will Data Be Saved?

| Operation | Result |
|---|---|
| df.show() | Displays data only |
| createOrReplaceTempView() | Creates a session-scoped SQL view; does not persist an independent table |
| df.write.csv(path) | Writes CSV files to the specified accessible path |
| df.write.parquet(path) | Writes Parquet files to the specified accessible path |
| df.write.format("delta").save(path) | Writes a Delta dataset to the specified path |
| df.write.saveAsTable(name) | Writes a registered table |

For a managed Unity Catalog table, Databricks manages its physical storage location.

A Unity Catalog Volume uses a path following this structure:

```text
/Volumes/catalog/schema/volume/folder/
```

### Current Practical Status

Our Cell 2 simulated write-mode behaviour using DataFrames.

Our Cell 3 displayed writing syntax and demonstrated processing partition changes.

Neither cell executed an actual persistent CSV, Parquet or Delta write.

We have confirmed that `workspace.default` exists, but have not yet verified its table-creation permissions or existing table names.

---

## 9. MERGE — Upcoming Topic

MERGE is not a standard DataFrameWriter write mode.

Delta MERGE supports conditional:

- INSERT
- UPDATE
- DELETE

We will cover MERGE separately during Delta Lake and incremental-loading practicals, including duplicate handling and pipeline restart scenarios.

---

## 10. Final Interview Takeaways

**Q1. What is the difference between append and overwrite?**

Append adds incoming records. Overwrite replaces existing data within the operation's target scope.

**Q2. Does append automatically prevent duplicates?**

No. Additional incremental-loading, deduplication or idempotency controls may be required.

**Q3. What is the difference between save() and saveAsTable()?**

`save()` writes to a specified path, whereas `saveAsTable()` creates or writes a registered table.

**Q4. Does coalesce(1) guarantee one output file?**

Not universally. Actual output layout depends on the writing operation and its configuration.

**Q5. What is the difference between repartition() and partitionBy()?**

`repartition()` changes DataFrame execution partitioning. `partitionBy()` organises data when writing it to storage.

**Q6. Is MERGE one of the four write modes?**

No. It is a separate conditional operation commonly used for Delta Lake upserts.

---

## Final Learning Principle

**Format** determines how data is stored.

**Mode** determines how existing target data is handled.

**Options** configure writing behaviour.

**Partitioning** influences processing or physical storage layout, depending on the operation.

**Catalog, schema and permissions** determine where registered tables can be created and accessed.